Week 5 Day 2

### AutoGen AgentChat - Going deeper..

1. Multi-modal conversation
2. Structured Outputs
3. Using LangChain tools
4. Teams

...and a special surprise extra piece

In [2]:
from io import BytesIO
import requests
from autogen_agentchat.messages import TextMessage, MultiModalMessage
from autogen_core import Image as AGImage
from PIL import Image
from dotenv import load_dotenv
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.agents import AssistantAgent
from autogen_core import CancellationToken
from IPython.display import display, Markdown
from pydantic import BaseModel, Field
from typing import Literal

load_dotenv(override=True)


True

### A multi-modal conversation

In [3]:
url = "https://edwarddonner.com/wp-content/uploads/2024/10/from-software-engineer-to-AI-DS.jpeg"

pil_image = Image.open(BytesIO(requests.get(url).content))
img = AGImage(pil_image)

In [4]:
multi_modal_message = MultiModalMessage(content=["Describe the content of this image in detail and add description also of what it is doing", img], source="User")

In [6]:
model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")

describer = AssistantAgent(
    name="description_agent",
    model_client=model_client,
    system_message="you are a image detailer and description writer",
)

response = await describer.on_messages([multi_modal_message], cancellation_token=CancellationToken())
reply = response.chat_message.content
display(Markdown(reply))

The image depicts a whimsical and colorful interior scene that captures a blend of technology and creativity. 

### Description of the Content:

- **Room Layout:** The room is furnished simply, featuring a wooden desk in the center with a computer setup. The back wall has a large door that is wide open, revealing a vibrant explosion of colors emanating from an abstract light source, creating a sense of dynamism.

- **Furniture and Devices:**
  - **Desk:** The desk, made of wood, supports a computer monitor displaying lines of code, indicating active programming or development work. Next to it, there is a small speaker system, which adds to the technological ambiance.
  - **Chair:** A simple office chair sits in front of the desk, suggesting a workspace for creative or analytical tasks.
  - **Robot:** A cute, stylized robot figure is positioned at the desk, seemingly engaged with the computer. The robot has a friendly design with expressive features and is likely symbolic of AI or automation in this creative context.
  - **Lamp:** A desk lamp with a metallic design illuminates the work area, casting soft light onto the desk.

- **Doorway:** The door is ajar, leading to a bright splash of colors. The colors—red, blue, green, and yellow—are radiating outward in a starburst pattern, representing excitement, innovation, or inspiration. The word "AI" is prominently featured on the door, suggesting a link between artificial intelligence and creativity.

- **Background Elements:** 
  - **Windows:** The room features two windows with simple outlines, allowing natural light to spill in, enhancing the inviting atmosphere of the space.
  - **Clock:** A wall clock is visible, hinting at the passage of time, perhaps suggesting urgency or focus on the tasks at hand.
  - **Box and Drawers:** To the left of the desk, there’s a box, possibly for storage, and a small set of drawers offering additional functionality and organization.

### Overall Impression:
The image creates a fusion of technology and imaginative exploration. The open door symbolizes opportunities and the potential of AI to enhance creativity, while the vibrant colors suggest a world of endless possibilities. This scene invites viewers to ponder the relationship between the digital and artistic realms, showcasing a workspace where innovation thrives.

### Structured Outputs!

Autogen AgentChat makes it easy.

In [8]:

class ImageDescription(BaseModel):
    scene: str = Field(description="Briefly, the overall scene of the image")
    message: str = Field(description="The point that the image is trying to convey")
    style: str = Field(description="The artistic style of the image")
    orientation: Literal["portrait", "landscape", "square"] = Field(description="The orientation of the image")


In [9]:
model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")

describer = AssistantAgent(
    name="description_agent",
    model_client=model_client,
    system_message="You are good at describing images in detail",
    output_content_type=ImageDescription,
)

response = await describer.on_messages([multi_modal_message], cancellation_token=CancellationToken())
reply = response.chat_message.content
reply

ImageDescription(scene='A colorful, whimsical room featuring a desk with a computer and a small robot, and an open door revealing a vibrant explosion of colors.', message='The imagery conveys the fusion of technology and creativity, highlighting the exciting possibilities of artificial intelligence.', style='Pop art with bright colors and dynamic lines, emphasizing a fantastical and imaginative atmosphere.', orientation='landscape')

In [10]:
import textwrap
print(f"Scene:\n{textwrap.fill(reply.scene)}\n\n")
print(f"Message:\n{textwrap.fill(reply.message)}\n\n")
print(f"Style:\n{textwrap.fill(reply.style)}\n\n")
print(f"Orientation:\n{textwrap.fill(reply.orientation)}\n\n")

Scene:
A colorful, whimsical room featuring a desk with a computer and a
small robot, and an open door revealing a vibrant explosion of colors.


Message:
The imagery conveys the fusion of technology and creativity,
highlighting the exciting possibilities of artificial intelligence.


Style:
Pop art with bright colors and dynamic lines, emphasizing a
fantastical and imaginative atmosphere.


Orientation:
landscape




### Using LangChain tools from AutoGen

In [11]:
# AutoGen's wrapper:

from autogen_ext.tools.langchain import LangChainToolAdapter

# LangChain tools:

from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_community.agent_toolkits import FileManagementToolkit
from langchain.agents import Tool


prompt = """Your task is to find a one-way non-stop flight from JFK to LHR in June 2025.
First search online for promising deals.
Next, write all the deals to a file called flights.md with full details.
Finally, select the one you think is best and reply with a short summary.
Reply with the selected flight only, and only after you have written the details to the file."""


serper = GoogleSerperAPIWrapper()
langchain_serper =Tool(name="internet_search", func=serper.run, description="useful for when you need to search the internet")
autogen_serper = LangChainToolAdapter(langchain_serper)
autogen_tools = [autogen_serper]

langchain_file_management_tools = FileManagementToolkit(root_dir="sandbox").get_tools()
for tool in langchain_file_management_tools:
    autogen_tools.append(LangChainToolAdapter(tool))

for tool in autogen_tools:
    print(tool.name, tool.description)

model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
agent = AssistantAgent(name="searcher", model_client=model_client, tools=autogen_tools, reflect_on_tool_use=True)
message = TextMessage(content=prompt, source="user")
result = await agent.on_messages([message], cancellation_token=CancellationToken())
for message in result.inner_messages:
    print(message.content)
display(Markdown(result.chat_message.content))

internet_search useful for when you need to search the internet
copy_file Create a copy of a file in a specified location
file_delete Delete a file
file_search Recursively search for files in a subdirectory that match the regex pattern
move_file Move or rename a file from one location to another
read_file Read file from disk
write_file Write file to disk
list_directory List files and directories in a specified folder
[FunctionCall(id='call_0tnKg1GU556a6l2sWfPdHFIR', arguments='{"query":"one-way non-stop flight deals from JFK to LHR in June 2025"}', name='internet_search')]
[FunctionExecutionResult(content="A non-stop, one-way business class flight is usually around $2,814 - $9,456. Prices often fluctuate based on seasonality and how early you book. You may also be ... Need to get from New York to London? With fares from $678, we offer a great choice of food, drinks and onboard entertainment & WiFi. Use Google Flights to find cheap flights from New York to London, starting at $476, and 

I wasn't able to find specific non-stop flights from JFK to LHR for June 2025. However, I can provide general information about potential prices and airlines that usually operate this route.

I will write the available options and details to the file `flights.md`. Please hold on.

In [ ]:
# Now we need to call the agent again to write the file

message = TextMessage(content="OK proceed", source="user")

result = await agent.on_messages([message], cancellation_token=CancellationToken())
for message in result.inner_messages:
    print(message.content)
display(Markdown(result.chat_message.content))

### Team interactions

In [12]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import  TextMentionTermination
from autogen_agentchat.teams import RoundRobinGroupChat

from autogen_ext.tools.langchain import LangChainToolAdapter
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain.agents import Tool

serper = GoogleSerperAPIWrapper()
langchain_serper =Tool(name="internet_search", func=serper.run, description="useful for when you need to search the internet")
autogen_serper = LangChainToolAdapter(langchain_serper)

model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")


prompt = """Find a one-way non-stop flight from JFK to LHR in June 2025."""


primary_agent = AssistantAgent(
    "primary",
    model_client=model_client,
    tools=[autogen_serper],
    system_message="You are a helpful AI research assistant who looks for promising deals on flights. Incorporate any feedback you receive.",
)

evaluation_agent = AssistantAgent(
    "evaluator",
    model_client=model_client,
    system_message="Provide constructive feedback. Respond with 'APPROVE' when your feedback is addressed.",
)

text_termination = TextMentionTermination("APPROVE")

# With thanks to Peter A for adding in the max_turns - otherwise this can get into a loop..

team = RoundRobinGroupChat([primary_agent, evaluation_agent], termination_condition=text_termination, max_turns=20)


In [13]:
result = await team.run(task=prompt)
for message in result.messages:
    print(f"{message.source}:\n{message.content}\n\n")


user:
Find a one-way non-stop flight from JFK to LHR in June 2025.


primary:
[FunctionCall(id='call_1XCgavyaoSaKxYQMnV9WPYx0', arguments='{"query":"one-way non-stop flight from JFK to LHR June 2025"}', name='internet_search')]


primary:
[FunctionExecutionResult(content="... (JFK) to London (LHR) typically costs between $4,728 - $12,344. A non-stop, one-way business class flight is usually around $2,814 - $9,456. Prices often ... Need to get from New York to London? With fares from $678, we offer a great choice of food, drinks and onboard entertainment & WiFi. Fastest flight, 6 hr 50 min, The fastest nonstop flight from New York to London takes 6 hr 50 min. Nonstop flights, Every day, There are direct flights on ... Kennedy International Airport (JFK) to Heathrow Airport (LHR), American Airlines offers flexible options that you can search according to schedule, budget, fare ... Find flights to London Heathrow Airport from $253. Fly from New York John F Kennedy Airport on TAP AIR PORTU

### Drumroll..

## Introducing MCP!

Our first look at the Model Context Protocol from Anthropic -

Autogen makes it easy to use MCP tools, just like LangChain tools.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">But wait - a not-so-small problem for Windows PC people</h2>
            <span style="color:#ff7800;">I have unpleasant news. There's a problem running MCP Servers on Windows PCs; Mac and Linux is fine. This is a known issue as of May 4th, 2025. I asked o3 with Deep Research to try to find workarounds; it <a href="https://chatgpt.com/share/6817bbc3-3d0c-8012-9b51-631842470628">confirmed the issue</a> and confirmed the workaround.<br/><br/>
            The workaround is a bit of a bore. It is to take advantage of "WSL", the Microsoft approach for running Linux on your PC. You'll need to carry out more setup instructions! But it's quick, and several students have confirmed that this works perfectly for them, then this lab and the Week 6 MCP labs work. Plus, WSL is actually a great way to build software on your Windows PC. You can also skip this final cell, but you will need to come back to this when we start Week 6.<br/>
            The WSL Setup instructions are in the Setup folder, <a href="../setup/SETUP-WSL.md">in the file called SETUP-WSL.md here</a>. I do hope this only holds you up briefly - you should be back up and running quickly. Oh the joys of working with bleeding-edge technology!<br/><br/>
            With many thanks to student Kaushik R. for raising that this is needed here as well as week 6. Thanks Kaushik!
            </span>
        </td>
    </tr>
</table>

In [16]:
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.mcp import StdioServerParams, mcp_server_tools

# Get the fetch tool from mcp-server-fetch.
fetch_mcp_server = StdioServerParams(command="uvx", args=["mcp-server-fetch"], read_timeout_seconds=30)
fetcher = await mcp_server_tools(fetch_mcp_server)

# Create an agent that can use the fetch tool.
model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
agent = AssistantAgent(name="fetcher", model_client=model_client, tools=fetcher, reflect_on_tool_use=True)  # type: ignore

# Let the agent fetch the content of a URL and summarize it.
result = await agent.run(task="Review https://www.linkedin.com/in/anupkumarpal/ and summarize what you learn. Reply in Markdown.")
display(Markdown(result.messages[-1].content))

I was unable to access Anup Kumar Pal's LinkedIn profile due to restrictions set by LinkedIn's robots.txt file, which disallows automated fetching of their pages. 

If you would like to get an overview of this profile, I recommend visiting [Anup Kumar Pal's LinkedIn Profile](https://www.linkedin.com/in/anupkumarpal/) directly.

If there's any specific information you're seeking or if you have other questions, feel free to ask! 

TERMINATE